# Min Fixation Rate
Determine the value of hyperparameter `MIN_FIXATION_RATE`, which is used to decide if a trial is valid or not (i.e., did the subject actually perform the task).<br><br>

<i>TLDR:</i> We set a value of `MIN_FIXATION_RATE = 2Hz` which is a round down from `mean - 3 * SD` fixation rate.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from statsmodels.sandbox.stats.stats_dhuard import percentileofscore

import config as cnfg
from analysis.trial_inclusion import actions

from plgrnd2 import fixations

pio.renderers.default = "notebook"
# pio.renderers.default = "browser"

### Read data

In [2]:
from analysis.helpers.read_data import read_data

loaded_data = read_data(cnfg.OUTPUT_PATH)
actions = loaded_data.actions
metadata = loaded_data.metadata
fixations = loaded_data.fixations
del loaded_data

### Calculate Fixation Rate

In [3]:
def calc_fixation_rate(fixs: pd.DataFrame, mdata: pd.DataFrame) -> pd.Series:
    """ Calculate fixation rate (fixations per second) for each trial. """
    n_fixs = fixs.groupby(["subject", "trial"]).size()
    trial_durations = mdata.set_index(["subject", "trial"])["duration"]     # in ms
    fix_rate = cnfg.MILLISECONDS_IN_SECOND * n_fixs / trial_durations       # convert to fixations per second
    return fix_rate

In [18]:
fix_rate = calc_fixation_rate(fixations, metadata)
fix_rate_summary = pd.concat([
    fix_rate.groupby("subject").median().rename("median"),
    fix_rate.groupby("subject").mean().rename("mean"),
    fix_rate.groupby("subject").std().rename("std"),
], axis=1)
fix_rate_summary.loc["all"] = (fix_rate.median(), fix_rate.mean(), fix_rate.std())
fix_rate_summary = fix_rate_summary.sort_index(
    key=lambda index: index.map(lambda subj: int(subj) if subj != "all" else -1 * float("inf"))
)
fix_rate_summary

,median,mean,std
subject,,,
all,4.349866,4.421202,0.682085
2,5.238075,5.265473,0.699074
12,4.789623,5.023316,0.850886
13,5.020108,5.036715,0.406634
14,3.850736,3.857977,0.286093
15,4.132538,4.019472,0.494722
16,4.005927,3.989283,0.277232
17,4.144458,4.186537,0.363473
18,4.574071,4.559364,0.276617


In [28]:
fix_rate_fig = go.Figure()
fix_rate_fig.add_trace(go.Scatter(
    x=[fix_rate_summary.loc["all", "mean"]],
    error_x=dict(type="data", array=[fix_rate_summary.loc["all", "std"]], visible=True,),
    y=pd.Series(0.5),
    name="Mean Fixation Rate", legendgroup="Mean",
    mode="markers",
    marker=dict(color=0, symbol="circle", size=10),
    text=["Mean Subject"],
    hoverinfo="x+y+text",
))
fix_rate_fig.add_trace(go.Scatter(
    x=fix_rate_summary.iloc[1:]["mean"],
    error_x=dict(type="data", array=fix_rate_summary.iloc[1:]["std"], visible=True,),
    y=np.random.rand(len(fix_rate_summary.iloc[1:])) * 0.5,
    name="Individual Fixation Rate", legendgroup="Mean",
    mode="markers",
    marker=dict(color=fix_rate_summary.iloc[1:].index, symbol="diamond", size=7.5),
    text=fix_rate_summary.iloc[1:].reset_index(drop=False).apply(
        lambda row: f"Subject: {row['subject']}<br>"
                    f"Median Fixation Rate: {row['median']:.2f} fix/s",
        axis=1
    ),
    hoverinfo="x+y+text",
))
fix_rate_fig.add_trace(go.Violin(
    x=fix_rate,
    name="Fixation Rate (fixs/s)", legendgroup="Fixation Rate",
    text=fix_rate.reset_index(drop=False).apply(
        lambda row: f"Subject: {row['subject']}<br>"
                    f"Trial: {row['trial']}<br>"
                    f"Fixation Rate: {row[0]:.2f} fix/s",
        axis=1
    ),
    width=1.75, orientation="h", side="positive", spanmode='hard',
    showlegend=True, hoverinfo="x+y+text",
    meanline=dict(visible=True),
    box=dict(visible=False), points=False,
))


fix_rate_fig.update_layout(
    title="Distribution of Fixation Rates Across Trials",
    xaxis_title="Fixation Rate (fixations per second)",
    yaxis_title="Density",
    violingap=0,
    violinmode="overlay",
)
fix_rate_fig.show()

### Check Different Thresholds

In [54]:
def _percentileofscore_safe(series: pd.Series, score: float) -> float:
    """ A safe version of percentileofscore that returns NaN if the series is empty. """
    if score < series.min():
        return 0.0
    if score > series.max():
        return 100.0
    return percentileofscore(series, score)

def calc_quantiles(threshold: float) -> pd.Series:
    name=f"quantile_at{threshold:.1f}"
    res = (
        fix_rate
        .groupby("subject")
        .apply(lambda x: _percentileofscore_safe(x, threshold))
        .rename(name)
    )
    res.loc["all"] = _percentileofscore_safe(fix_rate, threshold)
    res = res.sort_index(
        key=lambda index: index.map(lambda subj: int(subj) if subj != "all" else -1 * float("inf"))
    )
    return res

In [56]:
thresh = fix_rate_summary.loc["all", "mean"] - 2 * fix_rate_summary.loc["all", "std"]
global_rate_mean, global_rate_std = fix_rate_summary.loc["all", ["mean", "std"]]
thresholds = sorted([2, 3] + [round(global_rate_mean - k * global_rate_std, 1) for k in range(1, 4)])
pd.concat([calc_quantiles(threshold) for threshold in thresholds], axis=1)

,quantile_at2.0,quantile_at2.4,quantile_at3.0,quantile_at3.1,quantile_at3.7
subject,,,,,
all,0.0,0.0,0.943565,1.144279,11.528304
2,0.0,0.0,0.000000,0.000000,0.000000
12,0.0,0.0,0.000000,0.000000,0.000000
13,0.0,0.0,0.000000,0.000000,0.000000
14,0.0,0.0,0.000000,0.000000,25.679570
15,0.0,0.0,2.989443,4.948208,24.996064
16,0.0,0.0,0.000000,0.000000,14.272115
17,0.0,0.0,0.000000,0.000000,8.407313
18,0.0,0.0,0.000000,0.000000,0.000000


Based on the table above, there's not much difference between the `2`, `3`, and `mean - 3 * SD` threshold choices. We'll go with a simple `MIN_FIXATION_RATE = 2` fixations per second threshold for the main analysis.